# 🧠 Day 1 — Llion Jones Lab: 어텐션을 눈으로 본다

> 논문: **Attention Is All You Need** (Vaswani, …, Jones, … 2017)
> 핵심 한 줄: `Attention(Q,K,V) = softmax(QKᵀ/√d)·V` — *"각 단어가 다른 단어를 얼마나 쳐다보는가"*

구성: **Part 1** 손계산(numpy) → **Part 2** 진짜 모델 뇌 열어보기(transformer_lens) → **Part 3** 성격 조종(steering)

**런타임 → T4 GPU** 켜고, ①번부터 순서대로 ▶ (지난번 교훈: 중간부터 누르면 NameError!)

## ① 설치 (2~3분)

In [ ]:
%%capture
!pip install transformer_lens matplotlib

## Part 1 — 어텐션 손계산 🖐️
토큰 4개짜리 문장으로 Q·K·V를 직접 곱해본다. 논문 수식이 곧 이 몇 줄이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'  # 한글 라벨은 아래서 로마자 병기

np.random.seed(42)
tokens = ["나는", "식은", "커피를", "마셨다"]
labels = ["나는(I)", "식은(cold)", "커피를(coffee)", "마셨다(drank)"]
d = 8                                  # 임베딩 차원(작게)
X = np.random.randn(4, d)              # 각 토큰의 임베딩 (진짜 모델은 학습된 값)
Wq, Wk, Wv = (np.random.randn(d, d) for _ in range(3))

Q, K, V = X @ Wq, X @ Wk, X @ Wv       # ← 논문의 Q, K, V
scores = Q @ K.T / np.sqrt(d)          # QKᵀ/√d : '쳐다보는 정도'의 원점수

# GPT류는 인과(causal) 마스크: 미래 토큰은 못 봄 → 위삼각을 -inf
mask = np.triu(np.ones((4, 4)), k=1).astype(bool)
scores[mask] = -1e9

attn = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)  # softmax
out = attn @ V                         # 최종 출력 = 어텐션 가중 평균

print("어텐션 행렬 (행=지금 단어, 열=쳐다보는 대상, 행 합=1.0):")
print(np.round(attn, 2))

plt.figure(figsize=(5, 4))
plt.imshow(attn, cmap='Blues')
plt.xticks(range(4), labels, rotation=45, ha='right')
plt.yticks(range(4), labels)
plt.title('Hand-made Attention (causal)')
plt.colorbar(); plt.tight_layout(); plt.show()

**관찰 포인트**
- 각 행의 합 = 1.0 (softmax) — "주의력 100%를 어떻게 나눠 쓰나"
- 우상단이 흰색(0) = **미래는 못 봄** (인과 마스크)
- 진짜 모델과 차이는 딱 하나: W들이 *랜덤*이 아니라 *학습된 값*이라는 것.

---
## Part 2 — 진짜 모델의 뇌 열어보기 🔬 (transformer_lens)
코라 스타일 문장을 넣고, 레이어×헤드별로 누가 무엇을 쳐다보는지 본다.

In [ ]:
import torch
from transformer_lens import HookedTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'   # 코라와 같은 Qwen 계열(경량)
    model = HookedTransformer.from_pretrained(MODEL, device=device)
except Exception as e:
    print('Qwen 로드 실패 → gpt2로 폴백(영어 문장 사용):', e)
    MODEL = 'gpt2'
    model = HookedTransformer.from_pretrained(MODEL, device=device)
print(f'✅ {MODEL} | layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

In [ ]:
# 코라의 문장(비유가 이어지는 문장이 관찰 재미가 좋다)
prompt = '슬픔을 다 식은 국밥에 빗대면, 그제야 살아난다.'
if MODEL == 'gpt2':
    prompt = 'The cold soup made the sadness feel alive again.'

toks = model.to_str_tokens(prompt)
print(f'토큰 {len(toks)}개:', toks)
logits, cache = model.run_with_cache(prompt)

In [ ]:
# 레이어 3곳(앞/중간/뒤) × 헤드 4개 어텐션 히트맵 그리드
L = model.cfg.n_layers
layers = [0, L // 2, L - 1]
heads = list(range(4))

fig, axes = plt.subplots(len(layers), len(heads), figsize=(4 * len(heads), 3.2 * len(layers)))
for i, layer in enumerate(layers):
    pattern = cache['pattern', layer][0]          # [head, query, key]
    for j, h in enumerate(heads):
        ax = axes[i][j]
        ax.imshow(pattern[h].cpu(), cmap='Blues')
        ax.set_title(f'L{layer} H{h}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f'Attention patterns — "{prompt[:20]}…"')
plt.tight_layout(); plt.show()

print('관찰 가이드:')
print(' 1) 대각선 아래 한 칸이 진한 헤드 = 직전 토큰 헤드(prev-token head)')
print(' 2) 첫 열만 진한 헤드 = attention sink(할 일 없을 때 첫 토큰에 주의를 버려둠)')
print(' 3) 멀리 떨어진 특정 토큰을 콕 짚는 헤드 = 의미/비유를 잇는 후보 → 랩노트에 기록!')

In [ ]:
# 특정 헤드 확대 + 토큰 라벨 (위 그리드에서 흥미로운 L, H를 골라 바꿔보기)
LAYER, HEAD = L // 2, 0   # ← 여기를 바꿔가며 탐험
p = cache['pattern', LAYER][0, HEAD].cpu()
n = len(toks)
plt.figure(figsize=(0.6 * n + 2, 0.6 * n + 1))
plt.imshow(p, cmap='Blues')
plt.xticks(range(n), toks, rotation=90, fontsize=8)
plt.yticks(range(n), toks, fontsize=8)
plt.title(f'Layer {LAYER} · Head {HEAD}')
plt.colorbar(); plt.tight_layout(); plt.show()

---
## Part 3 — 성격 조종(activation steering) 🎛️
**파인튜닝 없이**, '차분한 문장 − 들뜬 문장'의 내부 활성 차이 벡터를 중간 레이어에 더해 말투를 기울인다.
(코라 LoRA가 *가중치*를 바꿨다면, steering은 *실행 중 신호*에 손을 대는 것)

In [ ]:
calm = ['고요한 저녁, 마음이 천천히 가라앉는다.', '잔잔한 호수처럼 차분하게 숨을 고른다.', '느리게, 담담하게 하루를 접는다.']
excited = ['대박!! 완전 신난다 최고야!!', '와 미쳤다 진짜 흥분돼서 잠이 안 와!!', '가자!! 오늘 텐션 끝까지 올린다!!']
if MODEL == 'gpt2':
    calm = ['The evening is calm and quiet and still.', 'Slowly and gently, the day settles down.', 'A peaceful, soft, serene night.']
    excited = ['WOW this is AMAZING let us GO!!', 'So hyped, totally thrilled, super excited!!', 'YES!! Best day ever, so pumped!!']

STEER_LAYER = model.cfg.n_layers // 2
hook_name = f'blocks.{STEER_LAYER}.hook_resid_post'

def mean_resid(prompts):
    accum = []
    for pr in prompts:
        _, c = model.run_with_cache(pr)
        accum.append(c[hook_name][0, -1])   # 마지막 토큰 위치의 residual
    return torch.stack(accum).mean(0)

vec = mean_resid(calm) - mean_resid(excited)   # '차분함 방향' 벡터
vec = vec / vec.norm()
print(f'✅ steering 벡터 완성 (layer {STEER_LAYER}, dim {vec.shape[0]})')

In [ ]:
test_prompt = '오늘 하루를 한 문장으로 말하면,'
if MODEL == 'gpt2':
    test_prompt = 'If I describe today in one sentence,'

def generate_with_alpha(alpha, max_new_tokens=40):
    def steer(resid, hook):
        return resid + alpha * vec * resid.norm(dim=-1, keepdim=True) * 0.1
    with model.hooks(fwd_hooks=[(hook_name, steer)]):
        return model.generate(test_prompt, max_new_tokens=max_new_tokens,
                              temperature=0.7, verbose=False)

for alpha in (+6, 0, -6):   # +차분 방향 / 원본 / -차분(=들뜸) 방향
    label = {6: '😌 +차분', 0: '⚪ 원본', -6: '🔥 -차분(들뜸)'}[alpha]
    print(f'\n[{label}]')
    print(generate_with_alpha(alpha))

**해석**
- 세 출력의 *온도 차이*가 느껴지면 성공 — 가중치를 1도 안 바꾸고 성격이 기울었다.
- 미묘하면 `alpha`(±4~±12)나 `STEER_LAYER`(±2)를 바꿔 재실행. 가끔 문장이 깨지는 것도 정상 관찰(과하게 밀면 뇌가 어지러워짐).
- 0.5B 소형 모델이라 효과가 은은할 수 있음 — *방향이 있다는 것*을 확인하는 게 목표.

---
## ✅ 오늘의 랩노트 (완료 조건)
1. Part 1 히트맵 1장 + Part 2 히트맵 2장(전체 그리드/확대) 저장 → `practice/labs/jones-attention.md`에 붙이기
2. "흥미로운 헤드" 최소 1개: **L?·H?가 무엇을 쳐다보더라** 한 문장 기록
3. steering 전/후 출력 1쌍 붙여넣기 + 느낀 점 한 줄

> 내일(Day 2)은 David Ha — 이 '뇌'들을 **mergekit으로 합치고 진화**시킨다. 오늘 이해한 레이어 구조가 그대로 재료가 된다. 🧬